# 🔍 IP OSINT & Fraud Investigation Toolkit
**Author:** Naim | Cybersecurity Portfolio Project

This notebook performs deep enrichment on a target IP address using multiple threat intelligence sources:
- 📍 Geolocation (city, country, lat/long)
- 🛡️ Abuse reputation (AbuseIPDB)
- 🌐 ISP / ASN info (VPN/datacenter detection)
- 🔎 Shodan — open ports & exposed services
- 📋 WHOIS — domain registration
- 🗺️ Visual map output (Folium)

> ⚠️ **For educational and authorized investigation use only.**

## 📦 1. Install Dependencies

In [ ]:
!pip install shodan ipwhois folium requests pandas --quiet
print('✅ All dependencies installed.')

## 🔑 2. API Keys Configuration
Get free API keys from:
- [ipinfo.io](https://ipinfo.io/signup) — free tier: 50k req/month
- [AbuseIPDB](https://www.abuseipdb.com/register) — free tier: 1k req/day
- [Shodan](https://account.shodan.io/register) — free tier available

In [ ]:
# ── API Keys ──────────────────────────────────────────────
IPINFO_TOKEN   = "YOUR_IPINFO_TOKEN"      # ipinfo.io
ABUSEIPDB_KEY  = "YOUR_ABUSEIPDB_KEY"     # abuseipdb.com
SHODAN_API_KEY = "YOUR_SHODAN_API_KEY"    # shodan.io

# ── Target IP ─────────────────────────────────────────────
TARGET_IP = "8.8.8.8"   # 🔁 Replace with the IP you want to investigate

print(f'🎯 Target IP: {TARGET_IP}')

## 📍 3. Geolocation — ipinfo.io

In [ ]:
import requests
import json

def get_geolocation(ip, token):
    url = f"https://ipinfo.io/{ip}?token={token}"
    r = requests.get(url, timeout=10)
    if r.status_code == 200:
        return r.json()
    else:
        return {"error": f"HTTP {r.status_code}"}

geo = get_geolocation(TARGET_IP, IPINFO_TOKEN)

print("\n📍 GEOLOCATION RESULT")
print("-" * 40)
for k, v in geo.items():
    print(f"  {k:<15}: {v}")

# Parse lat/long for map
lat, lon = None, None
if 'loc' in geo:
    lat, lon = map(float, geo['loc'].split(','))
    print(f"\n  📌 Coordinates: {lat}, {lon}")

## 🛡️ 4. Abuse Reputation — AbuseIPDB

In [ ]:
def get_abuse_report(ip, api_key):
    url = "https://api.abuseipdb.com/api/v2/check"
    headers = {"Key": api_key, "Accept": "application/json"}
    params  = {"ipAddress": ip, "maxAgeInDays": 90, "verbose": True}
    r = requests.get(url, headers=headers, params=params, timeout=10)
    if r.status_code == 200:
        return r.json().get('data', {})
    else:
        return {"error": f"HTTP {r.status_code}"}

abuse = get_abuse_report(TARGET_IP, ABUSEIPDB_KEY)

print("\n🛡️  ABUSE REPUTATION RESULT")
print("-" * 40)

score = abuse.get('abuseConfidenceScore', 'N/A')
total = abuse.get('totalReports', 'N/A')
last  = abuse.get('lastReportedAt', 'Never')
isvpn = abuse.get('isPublicProxy', False) or abuse.get('usageType', '') in ['VPN', 'Data Center/Web Hosting/Transit']

print(f"  Abuse Score     : {score}/100")
print(f"  Total Reports   : {total}")
print(f"  Last Reported   : {last}")
print(f"  Usage Type      : {abuse.get('usageType', 'N/A')}")
print(f"  ISP             : {abuse.get('isp', 'N/A')}")
print(f"  VPN/Proxy Flag  : {'⚠️ YES' if isvpn else '✅ No'}")

if isinstance(score, int):
    if score >= 80:
        print(f"\n  🔴 HIGH RISK — Likely malicious")
    elif score >= 40:
        print(f"\n  🟡 MEDIUM RISK — Suspicious activity reported")
    else:
        print(f"\n  🟢 LOW RISK — Clean or minimal reports")

## 🔎 5. Shodan — Open Ports & Exposed Services

In [ ]:
import shodan

def get_shodan_info(ip, api_key):
    try:
        api = shodan.Shodan(api_key)
        return api.host(ip)
    except shodan.APIError as e:
        return {"error": str(e)}

shodan_data = get_shodan_info(TARGET_IP, SHODAN_API_KEY)

print("\n🔎 SHODAN RESULT")
print("-" * 40)

if 'error' in shodan_data:
    print(f"  ⚠️  {shodan_data['error']}")
else:
    print(f"  OS              : {shodan_data.get('os', 'Unknown')}")
    print(f"  Open Ports      : {shodan_data.get('ports', [])}")
    print(f"  Last Seen       : {shodan_data.get('last_update', 'N/A')}")
    print(f"  Hostnames       : {shodan_data.get('hostnames', [])}")
    print(f"  Vulns           : {list(shodan_data.get('vulns', {}).keys()) or 'None found'}")
    print("\n  Services:")
    for item in shodan_data.get('data', [])[:5]:
        port    = item.get('port', '?')
        product = item.get('product', 'Unknown')
        banner  = item.get('data', '')[:80].replace('\n', ' ')
        print(f"    Port {port:<6} | {product:<20} | {banner}")

## 📋 6. WHOIS Lookup

In [ ]:
from ipwhois import IPWhois

def get_whois(ip):
    try:
        obj = IPWhois(ip)
        return obj.lookup_rdap(depth=1)
    except Exception as e:
        return {"error": str(e)}

whois_data = get_whois(TARGET_IP)

print("\n📋 WHOIS / RDAP RESULT")
print("-" * 40)

if 'error' in whois_data:
    print(f"  ⚠️  {whois_data['error']}")
else:
    print(f"  ASN             : {whois_data.get('asn', 'N/A')}")
    print(f"  ASN CIDR        : {whois_data.get('asn_cidr', 'N/A')}")
    print(f"  ASN Country     : {whois_data.get('asn_country_code', 'N/A')}")
    print(f"  ASN Description : {whois_data.get('asn_description', 'N/A')}")
    net = whois_data.get('network', {})
    print(f"  Network Name    : {net.get('name', 'N/A')}")
    print(f"  Network Type    : {net.get('type', 'N/A')}")
    print(f"  CIDR Range      : {net.get('cidr', 'N/A')}")

## 🗺️ 7. Visual Map — Folium

In [ ]:
import folium
from IPython.display import display

if lat and lon:
    city    = geo.get('city', 'Unknown')
    country = geo.get('country', 'Unknown')
    org     = geo.get('org', 'Unknown')

    m = folium.Map(location=[lat, lon], zoom_start=10, tiles='CartoDB dark_matter')

    popup_html = f"""
    <b>🎯 Target IP:</b> {TARGET_IP}<br>
    <b>📍 Location:</b> {city}, {country}<br>
    <b>🏢 ISP/Org:</b> {org}<br>
    <b>🛡️ Abuse Score:</b> {score}/100<br>
    <b>⚠️ VPN/Proxy:</b> {'Yes' if isvpn else 'No'}
    """

    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"🎯 {TARGET_IP} — {city}, {country}",
        icon=folium.Icon(color='red', icon='crosshairs', prefix='fa')
    ).add_to(m)

    folium.Circle(
        location=[lat, lon],
        radius=5000,
        color='red',
        fill=True,
        fill_opacity=0.15
    ).add_to(m)

    m.save('ip_location_map.html')
    display(m)
    print('\n✅ Map saved as ip_location_map.html')
else:
    print('⚠️ No coordinates found — check your ipinfo token.')

## 📊 8. Summary Report

In [ ]:
import pandas as pd

summary = {
    'Field': [
        'Target IP', 'City', 'Region', 'Country', 'Coordinates',
        'ISP/Org', 'Abuse Score', 'Total Abuse Reports',
        'Last Reported', 'Usage Type', 'VPN/Proxy',
        'ASN', 'ASN Description', 'Open Ports'
    ],
    'Value': [
        TARGET_IP,
        geo.get('city', 'N/A'),
        geo.get('region', 'N/A'),
        geo.get('country', 'N/A'),
        geo.get('loc', 'N/A'),
        geo.get('org', 'N/A'),
        f"{score}/100",
        total,
        last,
        abuse.get('usageType', 'N/A'),
        'Yes ⚠️' if isvpn else 'No ✅',
        whois_data.get('asn', 'N/A') if 'error' not in whois_data else 'N/A',
        whois_data.get('asn_description', 'N/A') if 'error' not in whois_data else 'N/A',
        str(shodan_data.get('ports', 'N/A')) if 'error' not in shodan_data else 'N/A'
    ]
}

df = pd.DataFrame(summary)
df = df.set_index('Field')

print("\n📊 FULL INVESTIGATION SUMMARY")
print("=" * 50)
print(df.to_string())

# Save to CSV
df.to_csv(f'osint_report_{TARGET_IP.replace(".","_")}.csv')
print(f"\n✅ Report saved as osint_report_{TARGET_IP.replace('.','_')}.csv")